# Implementation, Training, and Use of U-Net for Brain Tumor Segmentation

we plan to implement the U-Net architecture in practice, train it on a real dataset, and finally use the trained model for brain tumor segmentation.



```text
Dataset
   ↓
Data Analysis
   ↓
Image / Mask Preparation
   ↓
Train / Validation Split
   ↓
Data Preprocessing
   ↓
Data Augmentation
   ↓
Class Imbalance Analysis
   ↓
Class Weight
   ↓
U-Net Architecture
   ↓
Model Compilation
   ↓
Training
   ↓
Evaluation
   ↓
Prediction
   ↓
Confidence Threshold
   ↓
Segmentation Mask
```



In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
from keras.models import Model
from keras.layers import Input, Conv2D, MaxPooling2D, Conv2DTranspose, concatenate
from sklearn.model_selection import train_test_split

# Load Dataset

# Dataset Introduction

To implement **U-Net**, we use a **Brain Tumor Segmentation** dataset.

The dataset contains **brain MRI images** along with their corresponding **tumor segmentation masks**.

The dataset consists of **4 classes**:

| Label | Meaning |
|------:|---------|
| 0 | Background / No Tumor |
| 1 | Tumor Type 1 |
| 2 | Tumor Type 2 |
| 3 | Tumor Type 3 |

An important point is that the dataset does **not** contain only images with tumors.

Class `0` represents images where **no tumor is present**.

Therefore, the mask corresponding to a healthy image contains only background pixels:

```text
Mask = 0
````

This is important because the model must learn that:

> If there is no tumor in the image, it should not predict any region as a tumor.

```
```


In [ ]:
DATASET_PATH = './Brain Tumor Segmentation Dataset/'

````
# Dataset Structure

The overall structure of the dataset is as follows:

```text
Brain Tumor Dataset
│
├── image
│   ├── 0
│   ├── 1
│   ├── 2
│   └── 3
│
└── mask
    ├── 0
    ├── 1
    ├── 2
    └── 3
````

The `image` directory contains the original MRI images.

The `mask` directory contains the corresponding segmentation masks for each image.

Therefore, each sample has three main pieces of information:

```text
Image Path
Mask Path
Label
```

For example:

```text
Image:
image/2/sample_001.jpg

Mask:
mask/2/sample_001.jpg

Label:
2
```

The correspondence between the **Image** and its **Mask** must be preserved when creating the dataset.

This is important because each input image must be paired with its **correct segmentation mask** during training.

```
```


---

In this project, we use a **Sparse Label Representation**.

Instead of creating a separate channel for each class, we store the **class index** for each pixel.

For example:

```text
0 → Background
1 → Tumor Type 1
2 → Tumor Type 2
3 → Tumor Type 3
````

As a result, the mask looks like this:

```text
0 0 0 0 0
0 0 1 1 0
0 1 1 2 0
0 0 2 2 0
0 0 0 0 0
```

Each pixel contains a single integer representing its class.

This representation is called **Sparse Categorical Representation**.

---

# Sparse vs. One-Hot

There are two main ways to represent labels in a segmentation task.

## Sparse

Each pixel contains only a single class index:

```text
0
1
2
3
````

For example:

```text
2 → Tumor Type 2
```

## One-Hot

For four classes, each pixel is represented by four values.

For example, a pixel belonging to **Class 2**:

```text
[0, 0, 1, 0]
```

In this project, we use the **Sparse Representation**.

Therefore, the loss function must also be compatible with sparse integer labels:

```text
Sparse Labels
      ↓
Sparse Categorical Loss
```

For a multi-class segmentation problem, a suitable choice is:

**Sparse Categorical Crossentropy**

```
```



In [ ]:
image_base_path = os.path.join(DATASET_PATH, 'image')
image_base_path

In [ ]:
mask_base_path = os.path.join(DATASET_PATH, 'mask')
mask_base_path

In [ ]:
class_dirs = [d for d in os.listdir(image_base_path) if os.path.isdir(os.path.join(image_base_path, d))]
class_dirs

In [ ]:
all_image_paths = []
all_mask_paths = []
all_labels = []

for i in range(len(class_dirs)):
    class_dir = class_dirs[i]

    image_folder = os.path.join(image_base_path, class_dir)
    mask_folder = os.path.join(mask_base_path, class_dir)

    label = int(class_dir)

    for file_name in os.listdir(image_folder):
        if file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
            img_path = os.path.join(image_folder, file_name)
            
            base_name, extension = os.path.splitext(file_name)
            mask_filename = f"{base_name}_m{extension}"
            mask_path = os.path.join(mask_folder, mask_filename)
            
            if os.path.exists(mask_path):
                all_image_paths.append(img_path)
                all_mask_paths.append(mask_path)
                all_labels.append(label)

In [ ]:
all_image_paths[0]

In [ ]:
all_mask_paths[0]

In [ ]:
all_labels[0]

In [ ]:
len(all_image_paths)

In [ ]:
train_images, val_images, train_masks, val_masks, train_labels, val_labels = train_test_split(
    all_image_paths, all_mask_paths, all_labels, test_size=0.1, random_state=42, stratify=all_labels
)

In [ ]:
len(train_images)

In [ ]:
len(val_images)

# Masks Pixel Check



Our masks are stored as **Grayscale** images.

Ideally, a segmentation mask should contain only specific pixel values corresponding to the defined classes.

In the simplest case of **Binary Segmentation**:

```text
0   → Background
255 → Object
````

However, in this project, we are working with **multiple classes**.

Therefore, before training, we need to inspect the pixel values of the masks and make sure they are valid and consistent with the expected class labels.

For example, if we find pixel values such as:

```text
0
127
255
```

we need to determine whether these values actually represent the intended labels or whether they are the result of a **dataset or preprocessing issue**.

This inspection is important because incorrect mask values can lead to incorrect training targets and negatively affect the segmentation performance of the model.

```
```


In [ ]:
problematic_masks = []
all_unique_values = set()

for mask_path in all_mask_paths:
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    
    unique_values = np.unique(mask)
    
    all_unique_values.update(unique_values)
    
    extra_values = set(unique_values) - {0, 255}
    
    if extra_values:
        problematic_masks.append((mask_path, unique_values))


print(f"Global unique pixel values found in all masks: {sorted(list(all_unique_values))}")

# Histogram Analysis

Another useful tool for analyzing segmentation masks is the **Histogram**.

A histogram shows how frequently each **pixel value** occurs across the dataset.

For example, if the histogram looks like this:

```text
Pixel Value
    │
    │ █
    │ █
    │ █
    │ █                         █
    └──────────────────────────────
      0                         255
````

we can see that the number of **Background pixels** is much higher than the number of pixels belonging to the **Tumor**.

This situation is very common in **Image Segmentation**, because the object of interest usually occupies only a small portion of the image.

In other words:

```text
Background >> Object
```

This leads to an important issue known as **Class Imbalance**, which we need to consider before training the U-Net model.

```
```


In [ ]:
all_pixels_list = []

for mask_path in all_mask_paths:
    mask_image = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask_image is not None:
        all_pixels_list.append(mask_image.ravel())

all_pixel_values = np.concatenate(all_pixels_list)

plt.figure(figsize=(12, 7))
plt.hist(all_pixel_values, bins=256, range=[0, 256], color='blue')
plt.title('Histogram of Pixel Values for ALL Masks in the Dataset')
plt.xlabel('Pixel Value (0=Black, 255=White)')
plt.ylabel('Total Number of Pixels (Frequency)')
plt.grid(True, alpha=0.5)
plt.savefig("all_masks_histogram.png")

In [ ]:
IMG_SIZE = (256, 256)
BATCH_SIZE = 16

In [ ]:
def load_and_preprocess_multiclass(image_path, mask_path, label):
    IMG_SIZE = (256, 256)
    
    img = tf.io.read_file(image_path)
    img = tf.image.decode_image(img, channels=1)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, IMG_SIZE)

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_image(mask, channels=1)
    
    mask = tf.where(mask > 128, tf.cast(label, tf.uint8), tf.cast(0, tf.uint8))
    
    mask = tf.image.resize(mask, IMG_SIZE, method='nearest')
    
    return img, mask

# Important Note on Augmentation for Segmentation

In **Image Segmentation**, we need to be very careful when applying **Data Augmentation**.

If we transform the image but do not apply the same transformation to the mask, the **Image and Mask will no longer be aligned**.

For example, if we shift the image:

```text
10 pixels → Right
````

the mask must undergo exactly the same transformation:

```text
10 pixels → Right
```

Therefore, **geometric augmentations** such as:

* Rotation
* Translation
* Zoom
* Flip

must be applied **simultaneously to both the Image and the Mask**.

On the other hand, transformations such as **Brightness**, **Contrast**, and some other image-intensity adjustments can be applied **only to the Image**, because they do not change the spatial location of the objects.

The key rule is:

> **Any augmentation that changes the spatial geometry must be applied identically to both the Image and the Mask.**

```
```


In [ ]:
def augment_photometric(image, mask):
  
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
    image = tf.clip_by_value(image, 0.0, 1.0)
    
    return image, mask



After preparing the required information, we use `tf.data.Dataset` to build the input pipeline.

Our initial information consists of:

```text
Image Path
Mask Path
Label
````

Then, using a **Mapping Function**, these paths are loaded and processed into:

```text
Image
Mask
```

Conceptually:

```text
(Image Path, Mask Path, Label)
             ↓
        load_process
             ↓
       (Image, Mask)
```

The `Label` is mainly used during the dataset preparation and analysis process, while the model receives the **Image** and its corresponding **Mask** as training data.

---

# Shuffle

For the **Training Dataset**, we should shuffle the samples.

This is especially important for this dataset because the images may initially be organized according to their class:

```text
Class 0
Class 0
Class 0
...
Class 1
Class 1
...
Class 2
...
Class 3
```

If we train the model without shuffling, the early batches may contain a disproportionate number of samples from a particular class.

Therefore:

```python
dataset.shuffle(...)
```

randomizes the order of the samples.

Conceptually:

```text
Original Order
      ↓
Class 0 → Class 0 → Class 0 → Class 1 → Class 1 → ...
      ↓
     Shuffle
      ↓
Class 2 → Class 0 → Class 3 → Class 1 → Class 0 → ...
```

This helps provide more diverse batches during training.

---

# Batch

For training the neural network, the data is provided to the model in **batches**.

For example:

```text
Batch Size = 16
```

This means that **16 images** are processed together in each training step.

The training process can be visualized as:

```text
Batch 1
   ↓
 Loss
   ↓
Update Weights

Batch 2
   ↓
 Loss
   ↓
Update Weights

Batch 3
   ↓
 Loss
   ↓
Update Weights
```

After processing all batches in an epoch, the training process reports aggregated performance metrics, such as the average loss across the batches.

---

# Prefetch

To improve the performance of the input pipeline, we can use:

```python
prefetch(tf.data.AUTOTUNE)
```

The idea behind **Prefetch** is that while the GPU is training on the current batch, the input pipeline can prepare the next batch in the background.

Conceptually:

```text
GPU:
Training Batch 1
        │
        ▼
Training Batch 2

CPU / Input Pipeline:
Prepare Batch 2
        │
        ▼
Prepare Batch 3
```

As a result, **data preparation and model training can overlap as much as possible**, reducing the time the GPU spends waiting for the next batch.

In practice, a typical TensorFlow input pipeline can look like:

```python
dataset = (
    dataset
    .shuffle(...)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
```

```
```


In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_masks, train_labels))
train_dataset = train_dataset.map(load_and_preprocess_multiclass, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.map(augment_photometric, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(buffer_size=1000).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

# imbalance Data


One of the important challenges in this dataset is **Class Imbalance**.

Suppose the number of pixels belonging to each class in the masks is as follows:

```text
Class 0 → 10,000,000 pixels
Class 1 →  1,000,000 pixels
Class 2 →    600,000 pixels
Class 3 →    300,000 pixels
````

In this case, **Class 0** is much more frequent than the other classes.

If the model focuses only on minimizing the **Loss**, it may become biased toward predicting the most frequent class.

For example, the model might predict most pixels as **Background** and still achieve a relatively high **Pixel Accuracy**.

However, this does not necessarily mean that the segmentation is good.

The model may perform poorly on the tumor classes, even though the overall accuracy appears high.

Therefore, **Class Imbalance** must be analyzed and addressed before training the U-Net model.

```
```


In [ ]:
class_counts = {0: 0, 1: 0, 2: 0, 3: 0}
total_pixels = 0

for images_batch, masks_batch in train_dataset:
    for mask in masks_batch:
        mask_np = mask.numpy()
        total_pixels += mask_np.size
        unique, counts = np.unique(mask_np, return_counts=True)
        for u, c in zip(unique, counts):
            if u in class_counts:
                class_counts[u] += c

In [ ]:
total_pixels

In [ ]:
class_counts

## Median Frequency Balancing 



One of the solutions to the **Class Imbalance** problem is using:

> **Class Weights**

The idea is simple:

```text
Higher Class Frequency
        ↓
   Lower Weight

Lower Class Frequency
        ↓
   Higher Weight
````

Therefore, during **Loss** calculation, incorrect predictions for minority classes have a greater impact on the overall loss.

This encourages the model to pay more attention to **underrepresented classes** instead of being biased toward the dominant Background class.

```
```




To calculate the class weights, we can use the **Median Frequency Balancing** method.

The main idea is:

```text
Weight(class) =
Median Frequency / Frequency(class)
````

Therefore, a class with a **lower frequency** receives a **higher weight**.

This makes the model pay more attention to minority classes during training.

Using the **median** instead of the mean can provide more balanced behavior when there are large differences between class frequencies or when some classes behave like **outliers**.

```
```


In [ ]:
class_frequencies = {cls: count / total_pixels for cls, count in class_counts.items() if count > 0}

frequencies = list(class_frequencies.values())
median_frequency = np.median(frequencies)

mfb_weights = {}
for cls, freq in class_frequencies.items():
    mfb_weights[cls] = median_frequency / freq

print(mfb_weights)

In [ ]:
val_dataset = tf.data.Dataset.from_tensor_slices((val_images, val_masks, val_labels))
val_dataset = val_dataset.map(load_and_preprocess_multiclass, num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
print(f"Train dataset: {train_dataset}")
print(f"Validation dataset: {val_dataset}")

# Masks sample

In [ ]:
for images, masks in train_dataset.take(2): 
    
    sample_image = images[0]
    sample_mask = masks[0]
    print(np.unique(sample_mask))


    plt.figure(figsize=(8, 4))
    
    plt.subplot(1, 2, 1)
    plt.title("Sample Image")
    plt.imshow(sample_image, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.title("Multi-class Mask")
    plt.imshow(np.squeeze(sample_mask), cmap='jet', vmin=0, vmax=3) 
    plt.axis('off')

    plt.show()

In [ ]:
sample_mask.shape

# U-NET


Now that the dataset is ready, we can move on to building the **U-Net model**.

The input to the model is:

```text
256 × 256 × 1
````

The output of the model will be:

```text
256 × 256 × 4
```

Why `4`?

Because we have **four segmentation classes**:

```text
0 → Background
1 → Tumor Type 1
2 → Tumor Type 2
3 → Tumor Type 3
```

Therefore, for every pixel, the model produces **4 class scores/probabilities**, one for each class.

Conceptually:

```text
Input Image
256 × 256 × 1
       ↓
     U-Net
       ↓
Output
256 × 256 × 4
```

For each pixel, the model determines which of the four classes is the most likely class.

```
```


In [ ]:
inputs = Input(shape=(256, 256, 1))


# Block 1: 256x256 -> 128x128
conv1 = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
conv1 = Conv2D(32, (3, 3), activation='relu', padding='same')(conv1)
pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)

# Block 2: 128x128 -> 64x64
conv2 = Conv2D(64, (3, 3), activation='relu', padding='same')(pool1)
conv2 = Conv2D(64, (3, 3), activation='relu', padding='same')(conv2)
pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)

# Block 3: 64x64 -> 32x32
conv3 = Conv2D(128, (3, 3), activation='relu', padding='same')(pool2)
conv3 = Conv2D(128, (3, 3), activation='relu', padding='same')(conv3)
pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)

# Block 4: 32x32 -> 16x16
conv4 = Conv2D(256, (3, 3), activation='relu', padding='same')(pool3)
conv4 = Conv2D(256, (3, 3), activation='relu', padding='same')(conv4)
pool4 = MaxPooling2D(pool_size=(2, 2))(conv4)


bottleneck = Conv2D(512, (3, 3), activation='relu', padding='same')(pool4)
bottleneck = Conv2D(512, (3, 3), activation='relu', padding='same')(bottleneck)


# Upsample 1: 16x16 -> 32x32
up5 = Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(bottleneck)
concat5 = concatenate([conv4, up5])
conv5 = Conv2D(256, (3, 3), activation='relu', padding='same')(concat5)
conv5 = Conv2D(256, (3, 3), activation='relu', padding='same')(conv5)

# Upsample 2: 32x32 -> 64x64
up6 = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(conv5)
concat6 = concatenate([conv3, up6])
conv6 = Conv2D(128, (3, 3), activation='relu', padding='same')(concat6)
conv6 = Conv2D(128, (3, 3), activation='relu', padding='same')(conv6)

# Upsample 3: 64x64 -> 128x128
up7 = Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(conv6)
concat7 = concatenate([conv2, up7])
conv7 = Conv2D(64, (3, 3), activation='relu', padding='same')(concat7)
conv7 = Conv2D(64, (3, 3), activation='relu', padding='same')(conv7)

# Upsample 4: 128x128 -> 256x256
up8 = Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(conv7)
concat8 = concatenate([conv1, up8])
conv8 = Conv2D(32, (3, 3), activation='relu', padding='same')(concat8)
conv8 = Conv2D(32, (3, 3), activation='relu', padding='same')(conv8)

outputs = Conv2D(4, (1, 1), activation='softmax')(conv8)

model = Model(inputs=[inputs], outputs=[outputs])

model.summary()

In [ ]:
sparse_mean_iou = tf.keras.metrics.MeanIoU(num_classes=4,sparse_y_pred=False)

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy', sparse_mean_iou])

In [ ]:
from tensorflow.keras import callbacks
callbacks = [
    callbacks.ModelCheckpoint('./bestmodel.keras',
                             monitor="val_loss",
                             verbose=0,            
                             save_best_only=True),
]

In [ ]:
history = model.fit(
    train_dataset,
    epochs=60,
    validation_data=val_dataset,
    class_weight=mfb_weights,
    callbacks=callbacks
)

In [ ]:
history_dict = history.history

In [ ]:
plt.plot(history_dict['loss'], label='Training Loss')
plt.plot(history_dict['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.plot(history.history['mean_io_u'], label='Training Mean IoU')
plt.plot(history.history['val_mean_io_u'], label='Validation Mean IoU')
plt.xlabel('Epochs')
plt.ylabel('Mean IoU')
plt.title('Training and Validation Mean IoU')
plt.legend()
plt.grid(True)
plt.show()

# Evaluation

In [ ]:
from tensorflow.keras.models import load_model

best_model = load_model('./bestmodel.keras',)

In [ ]:
sample_index = 50
sample_image_path = train_images[sample_index]
sample_mask_path = train_masks[sample_index]
sample_label = train_labels[sample_index]

img = tf.io.read_file(sample_image_path)
img = tf.image.decode_png(img, channels=1)
img = tf.image.convert_image_dtype(img, tf.float32)

IMG_SIZE = (256, 256)
img_resized = tf.image.resize(img, IMG_SIZE)

img_for_prediction = tf.expand_dims(img_resized, axis=0)


prediction = best_model.predict(img_for_prediction)
confidence_threshold = 0.99

max_probs = np.max(prediction, axis=-1)
predicted_labels = np.argmax(prediction, axis=-1)

predicted_mask = np.where(max_probs < confidence_threshold, 0, predicted_labels)

predicted_mask = np.squeeze(predicted_mask)


_, true_mask = load_and_preprocess_multiclass(sample_image_path, sample_mask_path, sample_label)
true_mask = np.squeeze(true_mask.numpy())


plt.figure(figsize=(15, 5))


plt.subplot(1, 3, 1)
plt.title("Original Image")
plt.imshow(img_resized,cmap='gray')
plt.axis('off')


plt.subplot(1, 3, 2)
plt.title("True Mask")
plt.imshow(true_mask, cmap='jet', vmin=0, vmax=3)
plt.axis('off')


plt.subplot(1, 3, 3)
plt.title(f"Predicted Mask (Threshold={confidence_threshold})")
plt.imshow(predicted_mask, cmap='jet', vmin=0, vmax=3)
plt.axis('off')

plt.tight_layout()
plt.savefig("prediction_sample_with_threshold.png")
plt.show()

print(f"Unique values in True Mask: {np.unique(true_mask)}")
print(f"Unique values in Predicted Mask (with threshold): {np.unique(predicted_mask)}")